In [2]:
print('Hello!')

Hello!


In [12]:
!az ml datastore create --file my_blob_datastore.yml --resource-group "polandai-dev-01" --workspace-name "polandai-dev-01"

{
  "account_name": "polandaidevshared",
  "container_name": "dm-dp100-azureml",
  "credentials": {},
  "description": "Datastore pointing to a blob container.",
  "endpoint": "core.windows.net",
  "id": "/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandai-dev-01/providers/Microsoft.MachineLearningServices/workspaces/polandai-dev-01/datastores/dmdp100data",
  "name": "dmdp100data",
  "protocol": "https",
  "resourceGroup": "polandai-dev-01",
  "tags": {},
  "type": "azure_blob"
}


In [ ]:
asd

In [1]:
!az ml data create --type mltable --name "dm-dp100-diabetes-training" --path ./diabetes-data --datastore "dmdp100data" --resource-group "polandai-dev-01" --workspace-name "polandai-dev-01"

^C


{
  "creation_context": {
    "created_at": "2025-07-01T15:17:29.404056+00:00",
    "created_by": "Mika",
    "created_by_type": "User",
    "last_modified_at": "2025-07-01T15:17:29.448187+00:00"
  },
  "id": "/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandai-dev-01/providers/Microsoft.MachineLearningServices/workspaces/polandai-dev-01/data/dm-dp100-diabetes-training/versions/2",
  "name": "dm-dp100-diabetes-training",
  "path": "azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandai-dev-01/workspaces/polandai-dev-01/datastores/dmdp100data/paths/LocalUpload/7c5337d16d60092840ad67a80fef64b7/diabetes-data/",
  "properties": {},
  "resourceGroup": "polandai-dev-01",
  "tags": {},
  "type": "mltable",
  "version": "2"
}


In [ ]:
!az ml data create  --type uri_file  --name "dm-dp100-diabetes-data-file"  --path ./diabetes-data/diabetes.csv  --datastore "dmdp100data"  --resource-group "polandai-dev-01"  --workspace-name "polandai-dev-01"

{
  "creation_context": {
    "created_at": "2025-07-01T15:19:12.637573+00:00",
    "created_by": "Mika",
    "created_by_type": "User",
    "last_modified_at": "2025-07-01T15:19:12.653522+00:00"
  },
  "id": "/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandai-dev-01/providers/Microsoft.MachineLearningServices/workspaces/polandai-dev-01/data/dm-dp100-diabetes-data-file/versions/1",
  "name": "dm-dp100-diabetes-data-file",
  "path": "azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandai-dev-01/workspaces/polandai-dev-01/datastores/dmdp100data/paths/LocalUpload/a01a5b9f954664cdfd935246b25e7f69/diabetes.csv",
  "properties": {},
  "resourceGroup": "polandai-dev-01",
  "tags": {},
  "type": "uri_file",
  "version": "1"
}



Uploading diabetes.csv (< 1 MB): 0.00B [00:00, ?B/s]
Uploading diabetes.csv (< 1 MB): 100%|##########| 528k/528k [00:00<00:00, 2.12MB/s]
Uploading diabetes.csv (< 1 MB): 100%|##########| 528k/528k [00:00<00:00, 2.09MB/s]




: 

In [ ]:
%%writefile $script_folder/train-model-mlflow.py
# import libraries
import mlflow
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

def main(args):
    # read data
    df = get_data(args.training_data)

    # split data
    X_train, X_test, y_train, y_test = split_data(df)

    # train model
    model = train_model(args.reg_rate, X_train, X_test, y_train, y_test)

    # evaluate model
    eval_model(model, X_test, y_test)

# function that reads the data
def get_data(path):
    print("Reading data...")
    df = pd.read_csv(path)
    
    return df

# function that splits the data
def split_data(df):
    print("Splitting data...")
    X, y = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness',
    'SerumInsulin','BMI','DiabetesPedigree','Age']].values, df['Diabetic'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

    return X_train, X_test, y_train, y_test

# function that trains the model
def train_model(reg_rate, X_train, X_test, y_train, y_test):
    mlflow.log_param("Regularization rate", reg_rate)
    print("Training model...")
    model = LogisticRegression(C=1/reg_rate, solver="liblinear").fit(X_train, y_train)

    return model

# function that evaluates the model
def eval_model(model, X_test, y_test):
    # calculate accuracy
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    print('Accuracy:', acc)
    mlflow.log_metric("Accuracy", acc)

    # calculate AUC
    y_scores = model.predict_proba(X_test)
    auc = roc_auc_score(y_test,y_scores[:,1])
    print('AUC: ' + str(auc))
    mlflow.log_metric("AUC", auc)

    # plot ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:,1])
    fig = plt.figure(figsize=(6, 4))
    # Plot the diagonal 50% line
    plt.plot([0, 1], [0, 1], 'k--')
    # Plot the FPR and TPR achieved by our model
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.savefig("ROC-Curve.png")
    mlflow.log_artifact("ROC-Curve.png")    

def parse_args():
    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument("--training_data", dest='training_data',
                        type=str)
    parser.add_argument("--reg_rate", dest='reg_rate',
                        type=float, default=0.01)

    # parse args
    args = parser.parse_args()

    # return args
    return args

# run script
if __name__ == "__main__":
    # add space in logs
    print("\n\n")
    print("*" * 60)

    # parse args
    args = parse_args()

    # run main function
    main(args)

    # add space in logs
    print("*" * 60)
    print("\n\n")


In [17]:
!az ml data create --type mltable --name "dm-dp100-oj-training" --path ./orange-juice-data  --datastore "dmdp100data" --resource-group "polandai-dev-01" --workspace-name "polandai-dev-01"

{
  "creation_context": {
    "created_at": "2025-06-25T15:37:28.732601+00:00",
    "created_by": "Mika",
    "created_by_type": "User",
    "last_modified_at": "2025-06-25T15:37:28.745880+00:00"
  },
  "id": "/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandai-dev-01/providers/Microsoft.MachineLearningServices/workspaces/polandai-dev-01/data/dm-dp100-oj-training/versions/1",
  "name": "dm-dp100-oj-training",
  "path": "azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandai-dev-01/workspaces/polandai-dev-01/datastores/dmdp100data/paths/LocalUpload/4c578d3c628c6b5484fb216f0dad08ca/orange-juice-data/",
  "properties": {},
  "resourceGroup": "polandai-dev-01",
  "tags": {},
  "type": "mltable",
  "version": "1"
}



Uploading orange-juice-data (5.14 MBs): 100%|##########| 5140037/5140037 [00:00<00:00, 10704661.46it/s]


